# Distress Gesture Detection — Training Pipeline

**Before running any cell, complete these 3 setup steps:**

### 1. Enable GPU
Top-right → Settings (⚙) → Accelerator → **GPU T4 x2** → Save

### 2. Add Datasets (click + Add Input, search each name)
| Dataset | Author |
|---|---|
| `ur-fall-detection-dataset` | shahliza27 |
| `falldataset-imvia` | tuyenldvn |
| `skeleton-data-of-ntu-rgbd-60-dataset` | hungkhoi |
| `cctv-weapon-dataset` | simuletic |
| `cctv-atm-robbery-detection-dataset-gun-and-knife` | simuletic |
| `surveillance-vlm-weapon-and-knife-detection-dataset` | simuletic |

### 3. Turn on Internet
Settings (⚙) → Internet → **On**

---
**Run all cells top to bottom.**

## Cell 1 — Verify GPU + Clone Repo

In [ ]:
import subprocess, os, sys
from pathlib import Path

gpu = subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
print(f'GPU: {gpu}')

print('\nAttached datasets:')
for d in sorted(Path('/kaggle/input').iterdir()):
    print(f'  /kaggle/input/{d.name}')

REPO_URL = 'https://github.com/shrishri12062000/distress.git'
REPO_DIR = '/kaggle/working/distress-gesture-detection'

if Path(REPO_DIR).exists():
    print('\nRepo exists — pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    print('\nCloning repo...')
    ret = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if ret != 0:
        print('ERROR: Clone failed. Check Internet is ON in Settings.')

sys.path.insert(0, REPO_DIR)
print(f'\nRepo ready at: {REPO_DIR}')

## Cell 2 — Install Dependencies

In [ ]:
# Install ONLY packages not pre-installed on Kaggle
# Do NOT reinstall numpy/torch/opencv — Kaggle provides them
import subprocess
pkgs = ['mediapipe', 'datasets', 'ultralytics', 'onnx', 'onnxruntime', 'huggingface-hub']
for pkg in pkgs:
    ret = subprocess.run(f'pip install -q {pkg}', shell=True)
    print(f'  {pkg}: installed')

# Verify imports
import torch, cv2, mediapipe, ultralytics, onnx, onnxruntime
import numpy as np
print(f'\nnumpy:        {np.__version__}')
print(f'torch:        {torch.__version__}  (CUDA: {torch.cuda.is_available()})')
print(f'opencv:       {cv2.__version__}')
print(f'mediapipe:    {mediapipe.__version__}')
print(f'ultralytics:  {ultralytics.__version__}')
print(f'onnx:         {onnx.__version__}')
print('\nAll dependencies ready.')

## Cell 3 — Data Preparation
⏱ **30–90 minutes.** Do not close the tab.

In [ ]:
os.chdir(REPO_DIR)
%run kaggle/prepare_data.py

## Cell 4 — Validate Data

In [ ]:
%run kaggle/validate_data.py

## Cell 5 — Train ST-GCN
⏱ **2–4 hours on GPU.**

In [ ]:
%run kaggle/train_stgcn.py

## Cell 6 — Train YOLOv8n Knife Detector
⏱ **30–60 minutes.**

In [ ]:
%run kaggle/train_yolo.py

## Cell 7 — Export Models to ONNX

In [ ]:
%run kaggle/export_models.py

## Cell 8 — Verify + Download Models

In [ ]:
import numpy as np
import onnxruntime as ort
from pathlib import Path

MODELS = Path('/kaggle/working')

stgcn_path = MODELS / 'stgcn.onnx'
if stgcn_path.exists():
    sess  = ort.InferenceSession(str(stgcn_path), providers=['CPUExecutionProvider'])
    dummy = np.zeros((1, 3, 30, 17), dtype=np.float32)
    out   = sess.run(None, {sess.get_inputs()[0].name: dummy})[0][0]
    probs = np.exp(out) / np.exp(out).sum()
    print('ST-GCN output:')
    for name, p in zip(['normal','help_signal','collapse_falling','fall_down'], probs):
        print(f'  {name:<20}: {p:.4f}')
    print('  ST-GCN ONNX working')
else:
    print('ST-GCN ONNX not found')

yolo_path = MODELS / 'yolo_knife.onnx'
if yolo_path.exists():
    sess  = ort.InferenceSession(str(yolo_path), providers=['CPUExecutionProvider'])
    dummy = np.zeros((1, 3, 640, 640), dtype=np.float32)
    out   = sess.run(None, {sess.get_inputs()[0].name: dummy})
    print(f'YOLOv8 output shape: {out[0].shape} — working')
else:
    print('YOLOv8 ONNX not found')

print('\n' + '='*50)
print('DOWNLOAD: Right panel → Output → download:')
print('  stgcn.onnx  and  yolo_knife.onnx')
print('Place in: distress-gesture-detection/models/')
print('='*50)